<a href="https://colab.research.google.com/github/yolohe1210/cse450/blob/main/module6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup and imports


In [24]:
import json
import os
import random
import re
import string
import time

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

## Configuration


In [25]:
SEED = 450
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR = "module06_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RUN_AUTHORS = ["austen", "doyle", "twain"]
RNN_EPOCHS = 10
TRANSFORMER_EPOCHS = 10
SEQUENCE_LENGTH = 128
BATCH_SIZE = 64
VOCAB_LIMIT = 8192
EMBEDDING_DIM = 128
RNN_UNITS = 512
GENERATE_TOKENS = 200

PROMPT = (
    "The world seemed like such a peaceful place until the magic tree was "
    "discovered in London."
)

AUTHOR_URLS = {
    "austen": [
        "https://raw.githubusercontent.com/byui-cse/cse450-course/master/"
        "data/austen/austen.txt"
    ],
    "doyle": [
        "https://raw.githubusercontent.com/GITenberg/"
        "The-Adventures-of-Sherlock-Holmes_1661/master/1661.txt"
    ],
    "twain": [
        "https://raw.githubusercontent.com/hazelcast/"
        "hazelcast-jet/master/examples/wordcount/src/main/resources/books/"
        "tom-sawyer.txt"
    ],
}




## Text handling


In [26]:
def strip_gutenberg(text):
    """Remove the Project Gutenberg header and footer when present."""
    start = text.find("*** START OF")
    if start >= 0:
        text = text[text.find("\n", start) + 1:]
    end = text.find("*** END OF")
    if end >= 0:
        text = text[:end]
    return text


def preprocess_text(text):
    """Convert text into whitespace-separated word/punctuation tokens.

    Newlines are protected before whitespace normalization so paragraph
    boundaries are not silently destroyed.
    """
    text = strip_gutenberg(text)
    text = text.replace("\ufeff", "").replace("\r", "")
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("—", " - ").replace("–", " - ")
    text = text.replace("_", "")

    # Preserve line and tab information before collapsing ordinary spaces.
    text = text.replace("\n", " zznewlinezz ")
    text = text.replace("\t", " zztabzz ")
    text = re.sub(r" +", " zzspacezz ", text)

    # Mark sentence-initial capitals, then lowercase the rest.
    text = re.sub(
        r"(?<![a-zA-Z])([A-Z])",
        lambda match: "^" + match.group(1).lower(),
        text,
    )
    text = re.sub(r"([A-Z])", lambda match: match.group(1).lower(), text)

    for punctuation in string.punctuation:
        text = text.replace(punctuation, f" {punctuation} ")

    return re.sub(r" +", " ", text).strip()


def postprocess_text(text):
    """Convert generated tokens back into readable text."""
    text = text.replace("zztabzz", "\t")
    text = text.replace("zznewlinezz", "\n")
    text = text.replace("zzspacezz", " ")
    text = re.sub(r"\^([a-z])", lambda match: match.group(1).upper(), text)
    return text.replace("^", "")


def load_author_text(author):
    """Download and combine all books assigned to one author."""
    local_dir = os.path.join(OUTPUT_DIR, "books")
    os.makedirs(local_dir, exist_ok=True)
    raw_books = []

    for index, url in enumerate(AUTHOR_URLS[author]):
        filename = f"{author}_{index}.txt"
        local_path = os.path.join(local_dir, filename)
        if not os.path.exists(local_path):
            print(f"Downloading {url}")
            downloaded = tf.keras.utils.get_file(filename, url)
            with open(downloaded, "rb") as source, open(local_path, "wb") as target:
                target.write(source.read())

        with open(local_path, "rb") as file:
            raw_books.append(file.read().decode("utf-8", errors="ignore"))

    text = preprocess_text("\n".join(raw_books))
    print(f"{author}: {len(text.split()):,} tokens")
    return text




## Data setup


In [27]:
def make_vectorizer(text):
    vectorizer = layers.TextVectorization(
        standardize="lower",
        split="whitespace",
        max_tokens=VOCAB_LIMIT,
        output_mode="int",
    )
    vectorizer.adapt([text])
    vocabulary = vectorizer.get_vocabulary()
    return vectorizer, vocabulary


def split_input_target(sequence):
    return sequence[:-1], sequence[1:]


def make_dataset(text, vectorizer):
    token_ids = vectorizer(text)
    token_dataset = tf.data.Dataset.from_tensor_slices(token_ids)
    sequences = token_dataset.batch(SEQUENCE_LENGTH + 1, drop_remainder=True)
    dataset = sequences.map(split_input_target, num_parallel_calls=tf.data.AUTOTUNE)
    return (
        dataset
        .shuffle(10_000, seed=SEED, reshuffle_each_iteration=True)
        .batch(BATCH_SIZE, drop_remainder=True)
        .prefetch(tf.data.AUTOTUNE)
    )


def save_vocabulary(author, vocabulary):
    path = os.path.join(OUTPUT_DIR, f"{author}_vocabulary.txt")
    with open(path, "w", encoding="utf-8") as file:
        file.write("\n".join(str(word) for word in vocabulary))




## RNN


In [28]:
class AuthorTextModel(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim=128, rnn_units=512):
        super().__init__()
        self.embedding = layers.Embedding(vocab_size, embedding_dim)
        self.lstm1 = layers.LSTM(rnn_units, return_sequences=True, return_state=True)
        self.lstm2 = layers.LSTM(rnn_units, return_sequences=True, return_state=True)
        self.lstm3 = layers.LSTM(rnn_units, return_sequences=True, return_state=True)
        self.hidden1 = layers.Dense(embedding_dim * 8, activation="relu")
        self.hidden2 = layers.Dense(embedding_dim * 2, activation="relu")
        self.output_layer = layers.Dense(vocab_size)

    def call(self, inputs, states=None, return_state=False, training=False):
        x = self.embedding(inputs, training=training)
        batch_size = tf.shape(inputs)[0]

        if states is None:
            zero_state = lambda: [
                tf.zeros([batch_size, self.lstm1.units]),
                tf.zeros([batch_size, self.lstm1.units]),
            ]
            states = [zero_state(), zero_state(), zero_state()]

        outputs = []
        for lstm, state in zip((self.lstm1, self.lstm2, self.lstm3), states):
            x, hidden, cell = lstm(x, initial_state=state, training=training)
            outputs.append([hidden, cell])

        x = self.hidden1(x, training=training)
        x = self.hidden2(x, training=training)
        logits = self.output_layer(x, training=training)
        return (logits, outputs) if return_state else logits


def generate_rnn(model, vectorizer, vocabulary, prompt, temperature=0.7):
    """Generate tokens while preserving the model's recurrent state."""
    states = None
    current = tf.constant([preprocess_text(prompt)])
    result = [prompt]

    for _ in range(GENERATE_TOKENS):
        ids = vectorizer(current)
        logits, states = model(ids, states=states, return_state=True, training=False)
        logits = logits[:, -1, :] / temperature
        logits = tf.tensor_scatter_nd_update(
            logits,
            indices=[[0, 0], [0, 1]],
            updates=[-1e9, -1e9],
        )
        next_id = tf.random.categorical(logits, 1)[0, 0]
        next_word = str(vocabulary[int(next_id.numpy())])
        result.append(next_word)
        current = tf.constant([next_word])

    return postprocess_text("".join(result))




## Transformer


In [29]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embedding = layers.Embedding(vocab_size, embed_dim)
        self.position_embedding = layers.Embedding(maxlen, embed_dim)

    def call(self, inputs):
        positions = tf.range(start=0, limit=tf.shape(inputs)[-1], delta=1)
        return self.token_embedding(inputs) + self.position_embedding(positions)


class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, feed_forward_dim, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
        )
        self.feed_forward = keras.Sequential([
            layers.Dense(feed_forward_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False):
        attention_output = self.attention(
            inputs, inputs, use_causal_mask=True, training=training
        )
        x = self.norm1(inputs + self.dropout1(attention_output, training=training))
        feed_forward_output = self.feed_forward(x, training=training)
        return self.norm2(x + self.dropout2(feed_forward_output, training=training))


def build_transformer(vocab_size, embed_dim=128, num_heads=4,
                      feed_forward_dim=512, blocks=3):
    inputs = layers.Input(shape=(None,), dtype="int32")
    x = TokenAndPositionEmbedding(SEQUENCE_LENGTH, vocab_size, embed_dim)(inputs)
    for _ in range(blocks):
        x = TransformerBlock(embed_dim, num_heads, feed_forward_dim)(x)
    outputs = layers.Dense(vocab_size)(x)
    return keras.Model(inputs, outputs)


def generate_transformer(model, vectorizer, vocabulary, prompt, temperature=0.7):
    ids = vectorizer([preprocess_text(prompt)])[0].numpy().tolist()
    for _ in range(GENERATE_TOKENS):
        context = np.array([ids[-SEQUENCE_LENGTH:]], dtype=np.int32)
        logits = model(context, training=False)[0, -1].numpy().astype(np.float64)
        logits[:2] = -np.inf  # ID 0 is padding; ID 1 is [UNK].
        logits /= temperature
        logits -= np.max(logits)
        probabilities = np.exp(logits)
        probabilities /= probabilities.sum()
        ids.append(int(np.random.choice(len(probabilities), p=probabilities)))
    return postprocess_text("".join(str(vocabulary[index]) for index in ids))




## Training


In [30]:
def train_one_author(author):
    """Train both models for one author and save all artifacts."""
    text = load_author_text(author)
    vectorizer, vocabulary = make_vectorizer(text)
    save_vocabulary(author, vocabulary)
    dataset = make_dataset(text, vectorizer)
    vocab_size = len(vocabulary)
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    results = {"author": author, "vocab_size": vocab_size}

    # RNN/LSTM model.
    rnn = AuthorTextModel(vocab_size, EMBEDDING_DIM, RNN_UNITS)
    for input_batch, _ in dataset.take(1):
        rnn(input_batch)
    rnn.compile(optimizer=keras.optimizers.Adam(0.002), loss=loss)
    start = time.time()
    rnn_history = rnn.fit(dataset, epochs=RNN_EPOCHS, verbose=1)
    results["rnn_seconds_per_epoch"] = (time.time() - start) / RNN_EPOCHS
    results["rnn_final_loss"] = float(rnn_history.history["loss"][-1])
    rnn.save_weights(os.path.join(OUTPUT_DIR, f"{author}_rnn.weights.h5"))

    rnn_outputs = []
    for temperature in (0.5, 0.7, 0.9):
        generated = generate_rnn(rnn, vectorizer, vocabulary, PROMPT, temperature)
        rnn_outputs.append({"temperature": temperature, "text": generated})
    with open(os.path.join(OUTPUT_DIR, f"{author}_rnn_prompt.txt"), "w", encoding="utf-8") as file:
        for item in rnn_outputs:
            file.write(f"Temperature {item['temperature']}\n{item['text']}\n\n")

    # Transformer model.  It uses the same vectorizer and dataset for a fair comparison.
    transformer = build_transformer(vocab_size)
    transformer.compile(optimizer=keras.optimizers.Adam(0.001), loss=loss)
    start = time.time()
    transformer_history = transformer.fit(dataset, epochs=TRANSFORMER_EPOCHS, verbose=1)
    results["transformer_seconds_per_epoch"] = (time.time() - start) / TRANSFORMER_EPOCHS
    results["transformer_final_loss"] = float(transformer_history.history["loss"][-1])
    transformer.save_weights(os.path.join(OUTPUT_DIR, f"{author}_transformer.weights.h5"))

    transformer_outputs = []
    for temperature in (0.5, 0.7, 0.9):
        generated = generate_transformer(
            transformer, vectorizer, vocabulary, PROMPT, temperature
        )
        transformer_outputs.append({"temperature": temperature, "text": generated})
    with open(
        os.path.join(OUTPUT_DIR, f"{author}_transformer_prompt.txt"),
        "w",
        encoding="utf-8",
    ) as file:
        for item in transformer_outputs:
            file.write(f"Temperature {item['temperature']}\n{item['text']}\n\n")

    results["rnn_outputs"] = rnn_outputs
    results["transformer_outputs"] = transformer_outputs
    return results


all_results = []
for author in RUN_AUTHORS:
    print(f"\n========== {author.upper()} ==========")
    all_results.append(train_one_author(author))
    tf.keras.backend.clear_session()

with open(os.path.join(OUTPUT_DIR, "results.json"), "w", encoding="utf-8") as file:
    json.dump(all_results, file, indent=2)

print("\nSaved models, vocabulary files, prompt samples, and results.json in:")
print(os.path.abspath(OUTPUT_DIR))




========== AUSTEN ==========
austen: 2,207,671 tokens
Epoch 1/10
267/267 ━━━━━━━━━━━━━━━━━━━━ 54s 183ms/step - loss: 3.3190
Epoch 2/10
267/267 ━━━━━━━━━━━━━━━━━━━━ 50s 179ms/step - loss: 2.6725
Epoch 3/10
267/267 ━━━━━━━━━━━━━━━━━━━━ 50s 182ms/step - loss: 2.5133
Epoch 4/10
267/267 ━━━━━━━━━━━━━━━━━━━━ 51s 181ms/step - loss: 2.4489
Epoch 5/10
267/267 ━━━━━━━━━━━━━━━━━━━━ 50s 181ms/step - loss: 2.3821
Epoch 6/10
267/267 ━━━━━━━━━━━━━━━━━━━━ 50s 181ms/step - loss: 2.3212
Epoch 7/10
267/267 ━━━━━━━━━━━━━━━━━━━━ 50s 182ms/step - loss: 2.2732
Epoch 8/10
267/267 ━━━━━━━━━━━━━━━━━━━━ 50s 182ms/step - loss: 2.2304
Epoch 9/10
267/267 ━━━━━━━━━━━━━━━━━━━━ 50s 181ms/step - loss: 2.1975
Epoch 10/10
267/267 ━━━━━━━━━━━━━━━━━━━━ 50s 181ms/step - loss: 2.1662
Epoch 1/10
267/267 ━━━━━━━━━━━━━━━━━━━━ 29s 47ms/step - loss: 3.3765
Epoch 2/10
267/267 ━━━━━━━━━━━━━━━━━━━━ 15s 50ms/step - loss: 2.5613
Epoch 3/10
267/267 ━━━━━━━━━━━━━━━━━━━━ 15s 50ms/step - loss: 2.3355
Epoch 4/10
267/267 ━━━━━━━━━━━━━━━━━━

In [31]:
import glob
import os

for filename in sorted(glob.glob("/content/module06_outputs/*prompt.txt")):
    print("\n" + "=" * 80)
    print(os.path.basename(filename))
    print("=" * 80)

    with open(filename, encoding="utf-8") as file:
        print(file.read()[:3000])


austen_rnn_prompt.txt
Temperature 0.5
The world seemed like such a peaceful place until the magic tree was discovered in London. 
 She left her to be so much of the rest of the Park. 
 
 "I am sure you were a very great man in the house, I am sure you 
 know, I know you will not have done the matter a very bad person, 
 and I hope you must be a very good man--I am sure you will be so 
 much in the Campbells. You will not understand him to be a very 
 matter of a great deal of the letter.--You cannot 

Temperature 0.7
The world seemed like such a peaceful place until the magic tree was discovered in London. 
 The first letter of Marianne, however, the next morning was to be 
 called away by the drawing-room. It had been treated too much to 
 be in the Coles into the end of Miss Crawford's shop to Mansfield Park, 
 and Elinor could not meet her husband; for as soon as a clergyman 
 to her at first, from the most easy style of its fraternal 
 minutes his taste, and of a conversation of h